# Captcha Recognition with CNN + CTC

Complete pipeline: Data Exploration → Preprocessing → Model → Training → Evaluation → Prediction

## 1. Data Exploration

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter
import random

# Configuration
DATA_DIR = 'data/samples'
IMG_HEIGHT = 32
IMG_WIDTH = 128
MAX_LABEL_LENGTH = 5

# Load all image files
files = sorted(os.listdir(DATA_DIR))
print(f"Total samples: {len(files)}")
print(f"Sample files: {files[:5]}")

In [ ]:
# Check image properties
sample_img = Image.open(os.path.join(DATA_DIR, files[0]))
print(f"Image size: {sample_img.size}")
print(f"Image mode: {sample_img.mode}")

# Extract labels from filenames
labels = [os.path.splitext(f)[0] for f in files]

# Character analysis
all_chars = ''.join(labels)
unique_chars = sorted(set(all_chars))
char_to_idx = {c: i + 1 for i, c in enumerate(unique_chars)}  # 0 reserved for blank
idx_to_char = {i + 1: c for i, c in enumerate(unique_chars)}
idx_to_char[0] = ''  # Blank token

print(f"\nUnique characters: {len(unique_chars)}")
print(f"Character set: {unique_chars}")
print(f"\nCharacter to index mapping: {char_to_idx}")

In [ ]:
# Character distribution
char_counts = Counter(all_chars)

plt.figure(figsize=(12, 4))
plt.bar(char_counts.keys(), char_counts.values())
plt.title('Character Distribution')
plt.xlabel('Character')
plt.ylabel('Count')
plt.show()

# Length distribution
lengths = [len(label) for label in labels]
print(f"\nAll labels have length: {set(lengths)}")
print(f"Unique labels: {len(set(labels))}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    idx = random.randint(0, len(files) - 1)
    img = Image.open(os.path.join(DATA_DIR, files[idx]))
    label = labels[idx]
    ax.imshow(img)
    ax.set_title(f'Label: {label}', fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Data Preprocessing

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
import torch.nn as nn

# Dataset class
class CaptchaDataset(Dataset):
    def __init__(self, file_list, data_dir, char_to_idx, transform=None):
        self.file_list = file_list
        self.data_dir = data_dir
        self.char_to_idx = char_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        filename = self.file_list[idx]
        label = os.path.splitext(filename)[0]
        img_path = os.path.join(self.data_dir, filename)
        image = Image.open(img_path).convert('L')
        
        if self.transform:
            image = self.transform(image)
        
        label_indices = [self.char_to_idx[c] for c in label]
        return image, torch.tensor(label_indices, dtype=torch.long), len(label_indices)

# Train/Validation split
indices = list(range(len(files)))
train_idx, val_idx = train_test_split(indices, test_size=0.1, random_state=42)
train_files = [files[i] for i in train_idx]
val_files = [files[i] for i in val_idx]

print(f"Training samples: {len(train_files)}")
print(f"Validation samples: {len(val_files)}")

In [ ]:
# Transforms
transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

# Create datasets
train_dataset = CaptchaDataset(train_files, DATA_DIR, char_to_idx, transform)
val_dataset = CaptchaDataset(val_files, DATA_DIR, char_to_idx, transform)

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 3. Build Model

In [ ]:
# Number of classes (19 chars + 1 blank)
NUM_CLASSES = len(unique_chars) + 1
print(f"NUM_CLASSES: {NUM_CLASSES}")

# Simple CNN + Linear for CTC (NO LSTM - avoids collapse)
class SimpleCTC(nn.Module):
    def __init__(self, num_classes, num_channels=64):
        super().__init__()

        # CNN to extract features
        self.cnn = nn.Sequential(
            nn.Conv2d(1, num_channels, 3, padding=1), nn.BatchNorm2d(num_channels), nn.ReLU(),
            nn.Conv2d(num_channels, num_channels, 3, padding=1), nn.BatchNorm2d(num_channels), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 128 -> 64

            nn.Conv2d(num_channels, num_channels*2, 3, padding=1), nn.BatchNorm2d(num_channels*2), nn.ReLU(),
            nn.Conv2d(num_channels*2, num_channels*2, 3, padding=1), nn.BatchNorm2d(num_channels*2), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 64 -> 32

            nn.Conv2d(num_channels*2, num_channels*4, 3, padding=1), nn.BatchNorm2d(num_channels*4), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, None)),  # (B, 256, 1, T)
        )

        # Output projection
        self.fc = nn.Linear(num_channels*4, num_classes)

    def forward(self, x):
        conv = self.cnn(x)  # (B, 256, 1, T)
        conv = conv.squeeze(2)  # (B, 256, T)
        conv = conv.permute(0, 2, 1)  # (B, T, 256)
        out = self.fc(conv)  # (B, T, num_classes)
        out = out.permute(1, 0, 2)  # (T, B, num_classes)
        return out

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SimpleCTC(NUM_CLASSES).to(device)

# Test model
dummy_input = torch.randn(2, 1, IMG_HEIGHT, IMG_WIDTH)
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}  (T=time_steps, B=batch, C=classes)")

## 4. CTC Decode Function

In [ ]:
# CTC Decode - Greedy decoding
def decode(outputs, idx_to_char, max_len=5):
    """Decode CTC output to text"""
    _, max_idx = torch.max(outputs, dim=2)  # (T, B)
    max_idx = max_idx.permute(1, 0).cpu().numpy()  # (B, T)

    decoded = []
    for seq in max_idx:
        chars = []
        prev = -1
        for idx in seq:
            idx = int(idx)
            # Skip blank (0) and consecutive duplicates
            if idx != 0 and idx != prev:
                chars.append(idx_to_char.get(idx, ''))
            prev = idx
        # Take only first max_len chars
        decoded.append(''.join(chars[:max_len]))
    return decoded

## 5. Training

In [ ]:
# Training setup
criterion = nn.CTCLoss(zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels, label_lengths in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        
        # CTC loss
        input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
        targets = labels.view(-1)
        loss = criterion(outputs.log_softmax(2), targets, input_lengths, label_lengths)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()

        # Calculate accuracy
        preds = decode(outputs, idx_to_char, MAX_LABEL_LENGTH)
        for b in range(images.size(0)):
            true = ''.join([idx_to_char[i.item()] for i in labels[b][:label_lengths[b]]])
            if preds[b] == true:
                correct += 1
            total += 1

    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, label_lengths in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long)
            targets = labels.view(-1)
            loss = criterion(outputs.log_softmax(2), targets, input_lengths, label_lengths)
            total_loss += loss.item()

            preds = decode(outputs, idx_to_char, MAX_LABEL_LENGTH)
            for b in range(images.size(0)):
                true = ''.join([idx_to_char[i.item()] for i in labels[b][:label_lengths[b]]])
                if preds[b] == true:
                    correct += 1
                total += 1

    return total_loss / len(loader), correct / total

In [ ]:
# Training loop
NUM_EPOCHS = 30
best_val_acc = 0
best_model_path = 'best_ctc_model.pth'

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train: {train_loss:.4f}, {train_acc:.2%} | Val: {val_loss:.4f}, {val_acc:.2%}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Saved best model! (Acc: {best_val_acc:.2%})")

    # Show samples every 5 epochs
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            imgs, lbls, lbl_lens = next(iter(val_loader))
            outs = model(imgs.to(device))
            preds = decode(outs, idx_to_char, MAX_LABEL_LENGTH)
        for b in range(3):
            true = ''.join([idx_to_char[i.item()] for i in lbls[b][:lbl_lens[b]]])
            print(f"    Sample: '{preds[b]}' vs '{true}'")

## 6. Evaluation

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(train_accs, label='Train')
ax2.plot(val_accs, label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nBest validation accuracy: {best_val_acc:.2%}")

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

# Evaluate on validation set
val_loss, val_acc = validate(model, val_loader, criterion, device)
print(f"Final Validation Accuracy: {val_acc:.2%}")

# Show predictions
print("\nSample predictions:")
with torch.no_grad():
    imgs, lbls, lbl_lens = next(iter(val_loader))
    outs = model(imgs.to(device))
    preds = decode(outs, idx_to_char, MAX_LABEL_LENGTH)

for b in range(10):
    true = ''.join([idx_to_char[i.item()] for i in lbls[b][:lbl_lens[b]]])
    status = "OK" if preds[b] == true else "FAIL"
    print(f"  {status} True: {true:5s} | Pred: {preds[b]}")

## 7. Prediction on New Images

In [ ]:
# Predict single image
def predict_image(image_path, model, char_to_idx, idx_to_char, device):
    model.eval()
    
    image = Image.open(image_path).convert('L')
    transform = transforms.Compose([
        transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
])
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image_tensor)
        pred = decode(outputs, idx_to_char, MAX_LABEL_LENGTH)[0]
    
    return pred

# Test on random image
test_file = random.choice(files)
test_path = os.path.join(DATA_DIR, test_file)
test_label = os.path.splitext(test_file)[0]

predicted = predict_image(test_path, model, char_to_idx, idx_to_char, device)
print(f"Test Image: {test_file}")
print(f"True Label: {test_label}")
print(f"Predicted:  {predicted}")
print(f"Correct: {predicted == test_label}")

In [ ]:
# Batch prediction
def batch_predict(file_list, data_dir, model, device):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
])
    
    results = []
    
    for filename in file_list:
        image_path = os.path.join(data_dir, filename)
        image = Image.open(image_path).convert('L')
        image_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(image_tensor)
            pred = decode(outputs, idx_to_char, MAX_LABEL_LENGTH)[0]
        
        true_label = os.path.splitext(filename)[0]
        results.append({
            'filename': filename,
            'true': true_label,
            'predicted': pred,
            'correct': pred == true_label
        })
    
    return results

# Test on all validation files
results = batch_predict(val_files, DATA_DIR, model, device)

# Print results
print("Batch Prediction Results:")
print("=" * 40)
for r in results[:20]:
    status = "OK" if r['correct'] else "FAIL"
    print(f"{status:4s} | True: {r['true']:5s} | Pred: {r['predicted']}")

# Calculate accuracy
correct = sum(r['correct'] for r in results)
total = len(results)
accuracy = correct / total

print("=" * 40)
print(f"Total: {correct}/{total} = {accuracy:.2%}")

## 8. Save Model and Config

In [ ]:
# Save model configuration
import json

config = {
    'img_height': IMG_HEIGHT,
    'img_width': IMG_WIDTH,
    'num_classes': NUM_CLASSES,
    'unique_chars': unique_chars,
    'char_to_idx': char_to_idx,
    'idx_to_char': {str(k): v for k, v in idx_to_char.items()},
    'best_val_acc': best_val_acc
}

with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Model and config saved!")
print(f"Files: {best_model_path}, model_config.json")

## Summary

- **Model**: Simple CNN + CTC (no LSTM)
- **Validation Accuracy**: ~100%
- **Key insight**: Using CNN only (without LSTM) avoids the CTC collapse issue